# 04. Leakage-Controlled Train/Validation Splitting Protocol

This notebook partitions datasets using modular functions from `src/splits.py`:
- **Condition 1 (Random Split)**: Standard row-level site split (~95% protein overlap).
- **Condition 2 (Group Split by UniProt ID)**: Enforces strict separation by protein identifier (0% protein overlap).
- **Condition 3 (Cluster / Homology Split)**: Clusters sequences by identity to ensure no homologous proteins cross partitions.

Reports **Homology Leakage Diagnostics** on exact protein overlap and label balance.

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd

# Add project root to sys.path so src is importable in any environment (local, server, Colab)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.splits import get_split, audit_leakage
from src.data import LABEL_COLS

print(f"✓ Project root: {PROJECT_ROOT}")
print("✓ Successfully imported splitting utilities from src.splits")

✓ Project root: /home/emmy/ptmPred
✓ Successfully imported splitting utilities from src.splits


In [2]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Feature encoding to split: "default", "onehot", "blosum", "aapc", "hybrid"
ENCODING_MTHD = "onehot"

# Splitting strategy:
#   - "random":  Site-level split (severe protein leakage ~95%)
#   - "group":   Group split by UniProt ID (0% protein leakage)
#   - "cluster": Sequence similarity / Family cluster split
SPLIT_STRATEGY = "random"

TEST_SIZE = 0.20
RANDOM_STATE = 42

# Relative paths (fully portable across local machine, lab server, and Colab)
INPUT_FILE = PROJECT_ROOT / f"data_engineered/{ENCODING_MTHD}/train_with_features_{ENCODING_MTHD}.csv"
FALLBACK_FILE = PROJECT_ROOT / "data_engineered/train_with_features.csv"
OUTPUT_DIR = PROJECT_ROOT / f"data_engineered/{ENCODING_MTHD}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_OUTPUT = OUTPUT_DIR / f"train_with_features_{ENCODING_MTHD}_split.csv"
VAL_OUTPUT = OUTPUT_DIR / f"val_with_features_{ENCODING_MTHD}_split.csv"
TRAIN_TAGGED = OUTPUT_DIR / f"train_{ENCODING_MTHD}_{SPLIT_STRATEGY}_split.csv"
VAL_TAGGED = OUTPUT_DIR / f"val_{ENCODING_MTHD}_{SPLIT_STRATEGY}_split.csv"

print("Configuration:")
print(f"  Encoding:        {ENCODING_MTHD}")
print(f"  Split Strategy:  {SPLIT_STRATEGY}")
print(f"  Input Path:      {INPUT_FILE}")

Configuration:
  Encoding:        onehot
  Split Strategy:  random
  Input Path:      /home/emmy/ptmPred/data_engineered/onehot/train_with_features_onehot.csv


In [3]:
# Load dataset
if INPUT_FILE.exists():
    df = pd.read_csv(INPUT_FILE)
    print(f"✓ Loaded primary input: {INPUT_FILE}")
elif FALLBACK_FILE.exists():
    df = pd.read_csv(FALLBACK_FILE)
    print(f"✓ Loaded fallback input: {FALLBACK_FILE}")
else:
    raise FileNotFoundError(f"Neither {INPUT_FILE} nor {FALLBACK_FILE} exists.")

print(f"Total samples: {len(df):,}")
print(f"Total features: {len(df.columns)}")
print(f"Unique UniProt protein IDs: {df['ID'].nunique():,}")

✓ Loaded primary input: /home/emmy/ptmPred/data_engineered/onehot/train_with_features_onehot.csv
Total samples: 89,010
Total features: 116
Unique UniProt protein IDs: 9,414


/tmp/ipykernel_40255/950540960.py:3: DtypeWarning: Columns (74: cys_ctx_-1_Y, 95: cys_ctx_1_B) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_FILE)


In [4]:
# Execute split using src.splits
train_df, val_df = get_split(
    df,
    strategy=SPLIT_STRATEGY,
    test_size=TEST_SIZE,
    seed=RANDOM_STATE,
    group_col="ID"
)

# Audit Homology Leakage
diagnostics = audit_leakage(train_df, val_df, group_col="ID")

print("\n" + "="*70 + "\nHOMOLOGY LEAKAGE AUDIT\n" + "="*70)
for k, v in diagnostics.items():
    print(f"  {k:<25}: {v}")


HOMOLOGY LEAKAGE AUDIT
  train_samples            : 71208
  val_samples              : 17802
  train_unique_proteins    : 9154
  val_unique_proteins      : 6245
  overlapping_proteins     : 5985
  leakage_overlap_pct      : 95.84


In [5]:
# Save split datasets
train_df.to_csv(TRAIN_OUTPUT, index=False)
val_df.to_csv(VAL_OUTPUT, index=False)
train_df.to_csv(TRAIN_TAGGED, index=False)
val_df.to_csv(VAL_TAGGED, index=False)

print("\n" + "="*70 + "\nFILES SAVED\n" + "="*70)
print(f"✓ Saved primary training split:   {TRAIN_OUTPUT} ({len(train_df):,} rows)")
print(f"✓ Saved primary validation split: {VAL_OUTPUT} ({len(val_df):,} rows)")
print(f"✓ Saved tagged training split:    {TRAIN_TAGGED}")
print(f"✓ Saved tagged validation split:  {VAL_TAGGED}")


FILES SAVED
✓ Saved primary training split:   /home/emmy/ptmPred/data_engineered/onehot/train_with_features_onehot_split.csv (71,208 rows)
✓ Saved primary validation split: /home/emmy/ptmPred/data_engineered/onehot/val_with_features_onehot_split.csv (17,802 rows)
✓ Saved tagged training split:    /home/emmy/ptmPred/data_engineered/onehot/train_onehot_random_split.csv
✓ Saved tagged validation split:  /home/emmy/ptmPred/data_engineered/onehot/val_onehot_random_split.csv
